# Phase 5 Pair #4 — n=10 Graduation Retrain (Colab, v2)

**Sequence:** 1 → 6 setup, **7 = smoke (1 seed)**, 8 = verify smoke fired, **9 = full n=10**, 10 = aggregate, 11 = drill-down, 12 = copy to Drive.

**Why a smoke first.** The original n=10 attempt produced bit-identical κ=0 vs κ=2.0 results because two retrieval paths bypassed `update_metastability` (fixed in commit `c98c6ea`). The smoke is a 6–10 min check that the mechanism actually fires before committing ~30 min to all 10 seeds.

**Pre-committed pass criteria** (binding — no retuning if these fail):
1. Δ meta_stable_rate at W=3, bootstrap 95% CI upper < 0 AND mean ≤ −0.10
2. m_i CV > 0.1 within first eval (mechanism differentiating between atoms)
3. d_eff ≥ 25 at step 1800 (substrate non-regression — proxied by n_patterns)
4. D1 non-regression in `pair4_active`

**Configuration** (locked):
- μ_obs = 0.05  (m_i EMA halflife ≈ 14 retrievals)
- μ_rep = 0.5   (one replay drains m by half)
- κ_active = 2.0
- A+B substrate: alpha_anti=1.0, coverage_lambda=1.0, repulsion_step_size=100.0

**Colab gotchas:** parent kernel must NOT touch CUDA before workers launch; `--device cuda` is mandatory.

In [ ]:
# 1. Clone and verify the pair #4 fix is present (commit c98c6ea or later).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

import subprocess
markers = [
    ('def update_metastability',                    'src/energy_memory/phase4/consolidation.py', 'm_i EMA method'),
    ('metastability_obs_rate',                       'src/energy_memory/phase4/consolidation.py', 'config knob mu_obs'),
    ('metastability_gain',                           'src/energy_memory/phase4/replay_loop.py',   'priority gain kappa'),
    ('--metastability-obs-rate',                     'experiments/19_phase34_integrated.py',     'exp 19 CLI'),
    ('unit.consolidation.update_metastability',      'experiments/19_phase34_integrated.py',     'inlined exp 19 update'),
    ('self.consolidation.update_metastability',      'src/energy_memory/phase4/replay_loop.py',  'replay-cycle update'),
    ('weights_tensor=final_weights.detach()',        'src/energy_memory/memory/torch_hopfield.py','weights_tensor on result'),
    ('weights_tensor=final_weights.detach()',        'src/energy_memory/phase4/trajectory.py',    'weights_tensor on traced'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', '-e', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'{status:8s} {label}: {marker}')
    if r.stdout:
        print(f'  {r.stdout.strip().splitlines()[0]}')

In [ ]:
# 2. Mount Drive + stage the phase3c codebook.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets

In [ ]:
# 4. CPU-only sanity. Parent must NOT touch CUDA.
import sys, os, gc
sys.path.insert(0, 'src')

from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)
del cb; gc.collect()

# Inline smoke: instantiate the new pair #4 surface to catch wrong-checkout.
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
from energy_memory.phase4.replay_loop import ReplayConfig
import torch
cfg = ConsolidationConfig(m=4, metastability_obs_rate=0.05)
state = ConsolidationState(cfg, device='cpu')
for _ in range(3):
    state.add_pattern(novelty_strength=1.0)
state.update_metastability(torch.tensor([0.5, 0.3, 0.2]))
state.metastability_payback(0, factor=0.5)
assert state.metastability_ema.shape == (3,), 'metastability_ema shape wrong'
print('pair #4 inline smoke: PASS (m_i =', state.metastability_ema.tolist(), ')')

rc = ReplayConfig(metastability_gain=2.0, metastability_replay_decay=0.5)
assert rc.metastability_gain == 2.0 and rc.metastability_replay_decay == 0.5
print('ReplayConfig pair #4 fields: PASS')

In [ ]:
# 4b. Pre-warm the wikitext cache so subprocesses do not race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
del splits; gc.collect()
print('cache warmed.')

In [ ]:
# 4c. GPU info (still no CUDA init in parent).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

## STEP A — SMOKE (1 seed, ~6-10 min)

Confirms the pair #4 mechanism actually fires after the `c98c6ea` fix. If this passes, run cell 8 to verify, then cell 9 for the full n=10. If smoke shows m_i still zero or Δ still zero, stop and ping Claude — there is a deeper bug than what `c98c6ea` fixed.

In [ ]:
# 7 (SMOKE). 1 seed x 2 conditions in parallel. ~6-10 min.
import subprocess, os, time
from pathlib import Path

SEED = 17
N_CUES = 3000
RUN_TAG = 'phase5_pair4_smoke'

OBS_RATE = 0.05
REP_DECAY = 0.5
KAPPA_ACTIVE = 2.0
ALPHA_ANTI = 1.0
COVERAGE_LAMBDA = 1.0
COVERAGE_EMA_RATE = 0.01
REPULSION_STEP_SIZE = 100.0

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(condition_tag, kappa):
    out_dir = f'reports/{RUN_TAG}_{condition_tag}_seed{SEED}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{condition_tag}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/19_phase34_integrated.py',
        '--device', 'cuda', '--updater-kind', 'hebbian',
        '--seed', str(SEED), '--n-cues', str(N_CUES),
        '--store-threshold', '0.3',
        '--alpha-anti', str(ALPHA_ANTI),
        '--coverage-lambda', str(COVERAGE_LAMBDA),
        '--coverage-ema-rate', str(COVERAGE_EMA_RATE),
        '--repulsion-step-size', str(REPULSION_STEP_SIZE),
        '--metastability-obs-rate', str(OBS_RATE),
        '--metastability-gain', str(kappa),
        '--metastability-replay-decay', str(REP_DECAY),
        '--output-dir', out_dir,
    ]
    print(f'launching {condition_tag} (kappa={kappa}) -> {out_dir}')
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy()), logf

procs = [launch('kappa0_control', 0.0), launch('pair4_active', KAPPA_ACTIVE)]
t0 = time.time()
remaining = [0, 1]
while remaining:
    still = []
    for i in remaining:
        if procs[i][0].poll() is None:
            still.append(i)
        else:
            procs[i][1].close()
            print(f'  worker {i} done at {(time.time()-t0)/60:.1f} min')
    remaining = still
    if remaining:
        time.sleep(30)
print(f'SMOKE DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 8 (SMOKE VERIFY). Check the mechanism actually fired.
#
# What you want to see:
#   pair4_active:  m_mean > 0, m_std > 0  (mechanism is firing)
#   meta_stable_w3 different between conditions (mechanism affects training)
#
# If both conditions are bit-identical AGAIN: stop. Do NOT proceed to cell 9.
import json
SEED = 17
RUN_TAG = 'phase5_pair4_smoke'

results = {}
for tag in ('kappa0_control', 'pair4_active'):
    p = f'reports/{RUN_TAG}_{tag}_seed{SEED}/phase34_results.json'
    run = json.loads(open(p).read())
    pp = run['results']['phase3_phase4']
    last = max(pp, key=lambda r: r.get('cues_seen', 0))
    first = min(pp, key=lambda r: r.get('cues_seen', 0))
    dd_last = (last.get('death_diag') or {}).get('3', {})
    dd_first = (first.get('death_diag') or {}).get('3', {})
    results[tag] = {
        'ms_w3_final': last.get('meta_stable_w3'),
        'm_mean_first': dd_first.get('metastability_ema_mean', 0.0),
        'm_std_first':  dd_first.get('metastability_ema_std', 0.0),
        'm_mean_final': dd_last.get('metastability_ema_mean', 0.0),
        'm_max_final':  dd_last.get('metastability_ema_max', 0.0),
    }

print(f'=== Smoke result for seed {SEED} ===\n')
for tag, r in results.items():
    print(f'{tag}:')
    print(f'  meta_stable_w3 (final eval): {r["ms_w3_final"]:.4f}')
    print(f'  m_mean (first eval):  {r["m_mean_first"]:.4f}')
    print(f'  m_std  (first eval):  {r["m_std_first"]:.4f}')
    print(f'  m_mean (final eval):  {r["m_mean_final"]:.4f}')
    print(f'  m_max  (final eval):  {r["m_max_final"]:.4f}')
    print()

delta = results['pair4_active']['ms_w3_final'] - results['kappa0_control']['ms_w3_final']
m_max_active = results['pair4_active']['m_max_final']
print(f'Delta meta_stable_w3 (active - control) = {delta:+.4f}')
print(f'pair4_active m_max = {m_max_active:.4f}')

print()
fires = m_max_active > 1e-6
moves = abs(delta) > 1e-6
print(f'Mechanism fires (m_i > 0 in pair4_active): {"PASS" if fires else "FAIL - bug remains"}')
print(f'Mechanism affects training (delta != 0):    {"PASS" if moves else "FAIL - bug remains"}')
print()
if fires and moves:
    print('SMOKE PASS - proceed to cell 9 for the full n=10 run.')
else:
    print('SMOKE FAIL - DO NOT run cell 9. Ping Claude with these numbers.')

## STEP B — FULL n=10 (only if smoke passed)

~30 min for 10 seeds × `pair4_active`. We keep the seed-17 `pair4_active` from the smoke and reuse the κ=0 control data from the original buggy run (κ=0 is bit-identical regardless of the fix — verified by the fact that the priority composition collapses to baseline at κ=0). So we only need to run `pair4_active` for the other 9 seeds.

In [ ]:
# 9 (FULL n=10 pair4_active only). Re-uses original kappa0 control data from Drive.
import subprocess, os, time, shutil
from pathlib import Path

SEEDS = [11, 23, 1, 2, 3, 5, 7, 13, 29]   # seed 17 came from smoke
N_CUES = 3000
RUN_TAG = 'phase5_pair4_n10'

OBS_RATE = 0.05
REP_DECAY = 0.5
KAPPA_ACTIVE = 2.0
ALPHA_ANTI = 1.0
COVERAGE_LAMBDA = 1.0
COVERAGE_EMA_RATE = 0.01
REPULSION_STEP_SIZE = 100.0

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

# 9a. Pull the kappa0_control JSONs back from Drive (saved during the
#     original run). At kappa=0 the priority composition is bit-identical
#     to the pre-pivot baseline (verified by
#     tests/test_phase5_metastability.py::TestKappaZeroPreservesPriority),
#     so the original kappa0 numbers are still valid.
all_seeds = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
drive_src_root = '/content/drive/MyDrive/neuro-ai/results'
pulled = 0
missing = []
for s in all_seeds:
    src = f'{drive_src_root}/{RUN_TAG}_kappa0_control_seed{s}'
    dst = f'reports/{RUN_TAG}_kappa0_control_seed{s}'
    if os.path.isdir(src) and not os.path.isdir(dst):
        shutil.copytree(src, dst)
        pulled += 1
    elif not os.path.isdir(src):
        missing.append(s)
print(f'pulled {pulled} kappa0_control runs from Drive')
if missing:
    print(f'MISSING kappa0_control for seeds: {missing}')
    print('  (original run did not save these to Drive — would need to re-run them)')

# 9b. Stage the seed-17 pair4_active from the smoke into the n=10 layout
#     so the aggregator finds all 10 in one place.
smoke_src = 'reports/phase5_pair4_smoke_pair4_active_seed17'
n10_dst   = f'reports/{RUN_TAG}_pair4_active_seed17'
if os.path.isdir(smoke_src) and not os.path.isdir(n10_dst):
    shutil.copytree(smoke_src, n10_dst)
    print(f'staged smoke seed 17 -> {n10_dst}')

# 9c. Launch the remaining 9 pair4_active workers.
def launch(seed):
    out_dir = f'reports/{RUN_TAG}_pair4_active_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'pair4_active_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/19_phase34_integrated.py',
        '--device', 'cuda', '--updater-kind', 'hebbian',
        '--seed', str(seed), '--n-cues', str(N_CUES),
        '--store-threshold', '0.3',
        '--alpha-anti', str(ALPHA_ANTI),
        '--coverage-lambda', str(COVERAGE_LAMBDA),
        '--coverage-ema-rate', str(COVERAGE_EMA_RATE),
        '--repulsion-step-size', str(REPULSION_STEP_SIZE),
        '--metastability-obs-rate', str(OBS_RATE),
        '--metastability-gain', str(KAPPA_ACTIVE),
        '--metastability-replay-decay', str(REP_DECAY),
        '--output-dir', out_dir,
    ]
    print(f'launching pair4_active seed={seed} -> {out_dir}')
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy()), logf

procs = [launch(s) for s in SEEDS]
t0 = time.time()
remaining = list(range(len(procs)))
while remaining:
    still = []
    for i in remaining:
        if procs[i][0].poll() is None:
            still.append(i)
        else:
            procs[i][1].close()
            print(f'  worker (seed {SEEDS[i]}) done at {(time.time()-t0)/60:.1f} min')
    remaining = still
    if remaining:
        time.sleep(60)
print(f'FULL pair4_active n=9 DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 9b. EMERGENCY kill if cell 9 misbehaves. Interrupt cell 9 first, then run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  pid err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 10. Aggregate the headline: Delta meta_stable_rate at W=3 across same-seed pairs.
#
# kappa0_control data is read from the ORIGINAL n=10 run (bit-identical
# to a fresh kappa=0 run — at kappa=0 the priority composition collapses
# to the pre-pivot baseline regardless of whether update_metastability
# fires).
# pair4_active data is read from the freshly-fixed run (cells 7 + 9).
import json
import numpy as np
from pathlib import Path

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
RUN_TAG = 'phase5_pair4_n10'

def load_run(condition_tag, seed):
    p = Path(f'reports/{RUN_TAG}_{condition_tag}_seed{seed}/phase34_results.json')
    if not p.exists():
        return None
    return json.loads(p.read_text())

def w3_meta_stable_rate_at_last_eval(run):
    if run is None:
        return None
    rows = run.get('results', {}).get('phase3_phase4', [])
    if not rows:
        return None
    last = max(rows, key=lambda r: r.get('cues_seen', 0))
    return last.get('meta_stable_w3')

pairs = []
for s in SEEDS:
    ctl = load_run('kappa0_control', s)
    act = load_run('pair4_active', s)
    r_ctl = w3_meta_stable_rate_at_last_eval(ctl)
    r_act = w3_meta_stable_rate_at_last_eval(act)
    if r_ctl is None or r_act is None:
        print(f'seed {s}: MISSING (ctl={r_ctl}, act={r_act})')
        continue
    delta = r_act - r_ctl
    pairs.append((s, r_ctl, r_act, delta))
    print(f'seed {s}: kappa0={r_ctl:.4f}  pair4={r_act:.4f}  delta={delta:+.4f}')

deltas = np.array([p[3] for p in pairs])
print(f'\n=== Headline: Delta meta_stable_rate at W=3, n={len(deltas)} same-seed pairs ===')
print(f'  mean delta = {deltas.mean():+.4f}')
print(f'  std  delta = {deltas.std(ddof=1):.4f}')

rng = np.random.default_rng(2026)
boots = [rng.choice(deltas, size=len(deltas), replace=True).mean() for _ in range(10_000)]
ci_lo, ci_hi = np.quantile(boots, [0.025, 0.975])
print(f'  bootstrap 95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]')

PASS_THRESHOLD = -0.10
ci_disjoint_neg = ci_hi < 0.0
hits_threshold = deltas.mean() <= PASS_THRESHOLD
print()
print('PASS criteria:')
print(f'  bootstrap CI disjoint from zero (upper < 0): {"PASS" if ci_disjoint_neg else "FAIL"}')
print(f'  mean delta <= {PASS_THRESHOLD:.2f}:                       {"PASS" if hits_threshold else "FAIL"}')
print(f'  overall:                                       {"PASS" if (ci_disjoint_neg and hits_threshold) else "FAIL"}')

In [ ]:
# 11. Drill-downs: m_i distribution + n_patterns + D1 non-regression.
def first_eval_row(run):
    rows = (run or {}).get('results', {}).get('phase3_phase4', [])
    return min(rows, key=lambda r: r.get('cues_seen', 0)) if rows else None

def last_eval_row(run):
    rows = (run or {}).get('results', {}).get('phase3_phase4', [])
    return max(rows, key=lambda r: r.get('cues_seen', 0)) if rows else None

def death_diag_for_scale(row, scale):
    dd = (row or {}).get('death_diag', {}) or {}
    return dd.get(str(scale)) or dd.get(scale) or {}

print('=== m_i distribution (CV > 0.1 = mechanism differentiating between atoms) ===\n')
for s in SEEDS:
    act = load_run('pair4_active', s)
    if not act:
        continue
    row = first_eval_row(act)
    if not row:
        continue
    dd = death_diag_for_scale(row, 3)
    m_mean = dd.get('metastability_ema_mean', 0.0) or 0.0
    m_std = dd.get('metastability_ema_std', 0.0) or 0.0
    cv = (m_std / m_mean) if m_mean > 1e-6 else 0.0
    print(f'  seed {s}: m_mean={m_mean:.4f}  m_std={m_std:.4f}  CV={cv:.2f}  '
          f'({"PASS" if cv > 0.1 else "FAIL uniform"})')

print('\n=== Substrate survival (W=4 final n_patterns; proxy for d_eff) ===\n')
for s in SEEDS:
    act = load_run('pair4_active', s)
    if not act:
        continue
    row = last_eval_row(act)
    if not row:
        continue
    dd = death_diag_for_scale(row, 4)
    n_patterns = dd.get('n_patterns', None)
    print(f'  seed {s}: W=4 final n_patterns = {n_patterns}')

print('\n=== D1 non-regression: meta_stable_w3 final ===\n')
for s in SEEDS:
    ctl = load_run('kappa0_control', s)
    act = load_run('pair4_active', s)
    if not (ctl and act):
        continue
    ms_ctl = (last_eval_row(ctl) or {}).get('meta_stable_w3', None)
    ms_act = (last_eval_row(act) or {}).get('meta_stable_w3', None)
    if ms_ctl is None or ms_act is None:
        continue
    print(f'  seed {s}: kappa0 ms_w3={ms_ctl:.4f}  pair4 ms_w3={ms_act:.4f}')

In [ ]:
# 12. Copy results back to Drive.
import shutil, os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
RUN_TAG = 'phase5_pair4_n10'
SMOKE_TAG = 'phase5_pair4_smoke'
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)

# Smoke (1 seed both conditions)
for tag in ('kappa0_control', 'pair4_active'):
    src = f'reports/{SMOKE_TAG}_{tag}_seed17'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/{SMOKE_TAG}_{tag}_seed17', dirs_exist_ok=True)
shutil.copytree(f'reports/{SMOKE_TAG}_colab', f'{dst}/{SMOKE_TAG}_colab', dirs_exist_ok=True)

# Full n=10 (pair4_active for all seeds)
for s in SEEDS:
    src = f'reports/{RUN_TAG}_pair4_active_seed{s}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/{RUN_TAG}_pair4_active_seed{s}', dirs_exist_ok=True)
shutil.copytree(f'reports/{RUN_TAG}_colab', f'{dst}/{RUN_TAG}_colab', dirs_exist_ok=True)

print('results copied to', dst)
!ls {dst} | grep -E 'phase5_pair4' | head -25